In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

# Same root detection as nanowm-m3-profile: search only inside the code
# dataset (the latents dataset carries no project tree, but a stale one
# mounted from another kernel's output would otherwise win), shallowest
# match, then verify the file this kernel actually needs is there.
inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))
candidates = [p for p in inputs.rglob("src/train/trainer.py") if "nanowm-code" in str(p)]
if not candidates:
    raise RuntimeError(f"no nanowm-code tree under {inputs}")
mounted = sorted(candidates, key=lambda q: len(q.parts))[0].parent.parent.parent
print("mounted project root:", mounted)

root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
assert (root / "scripts" / "run_m3_ablation.py").exists(), sorted(p.name for p in root.iterdir())
assert (root / "results" / "m3_profile.json").exists(), "M3-A profile missing: steps_per_arm cannot be derived"
print("project root:", root)


In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers"], check=True)
# The full suite costs ~1 CPU-minute and no GPU budget; a failure here is
# the cheapest possible place to find one.
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)


In [ ]:
# CLAUDE.md: "Latents als Kaggle-Dataset ablegen, nicht neu berechnen."
# says43/nanowm-m3-latents: 160 trajectories WITH DINOv2-small features,
# rendered once by nanowm-m3-preflight. Same mount check as the profile
# kernel that measured the throughput this run's schedules rest on.
import numpy as np, json

matches = [p for p in Path("/kaggle/input").rglob("manifest.json") if "nanowm-m3-latents" in str(p)]
if len(matches) != 1:
    raise RuntimeError(f"expected exactly one precomputed m3 manifest, got {matches}")
DATA_DIR = matches[0].parent
manifest = json.load(open(matches[0]))
print("data dir:", DATA_DIR)
print("trajectories:", manifest["num_trajectories"], "frames:", manifest["total_frames"])
print("dinov2:", manifest["dinov2_model_id"], manifest["dinov2_feature_dim"])

sample = sorted(DATA_DIR.glob("*.npz"))[0]
with np.load(sample) as data:
    print(sample.name, {k: data[k].shape for k in data.files})
    assert data["dinov2"].shape == (data["latents"].shape[0], 16, 384)


In [ ]:
# configs/m3_ablation.yaml ships with ticket_hours, seconds_per_arm and
# every steps_per_arm entry null (tests/test_m3_ablation.py guards all
# three -- never commit live values). They are patched in only on this
# Kaggle copy, from the approved ticket and the M3-A profile.
#
# Ticket M3-B, approved 2026-09-18: 4.25 GPU-hours (single T4, as M2/M3-A).
#   2x2 matrix (REPA on/off x Muon vs AdamW) x 2 seeds = 8 arms
#   1800 s wall-clock per arm = 4.00 h of training
#   0.25 h margin for 8x model build + torch.compile + dataloader setup
#   (M3-A: ~17 s per compile; 8 x 17 s = 2.3 min, so this margin is generous)
#   Abort criterion: run_m3_ablation.py's own hard deadline at 4.25 h;
#   the JSON is snapshotted after every arm, seed-major, so a truncated
#   matrix still contains whole seeds.
#   Expected: paired REPA and Muon deltas with per-seed sign agreement.
#   Muon arms get ~55% of the AdamW arms' steps (measured 44% throughput
#   cost) -- that is the price the equal-wall-clock rule charges them.
#
# steps_per_arm comes from results/m3_profile.json (M3-A, this exact
# preset/batch/hardware, target_seconds_per_arm=1800, safety margin 0.9),
# NOT typed by hand, so the LR schedule horizon and the wall-clock slice
# are derived from the same measurement.
import yaml

TICKET_HOURS = 4.25
SECONDS_PER_ARM = 1800

profile = json.load(open("results/m3_profile.json"))
assert profile["complete"], "M3-A profile is incomplete"
assert profile["config"]["profile"]["target_seconds_per_arm"] == SECONDS_PER_ARM
assert profile["config"]["model"]["preset"] == "15m"
steps_per_arm = {k: int(v) for k, v in profile["recommended_steps_per_arm"].items()}
print("steps_per_arm from M3-A:", steps_per_arm)

cfg_path = Path("configs/m3_ablation.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
assert set(cfg["ablation"]["steps_per_arm"]) == set(steps_per_arm), "arm names drifted"
assert cfg["optim"]["muon_lr"] == 0.02, "muon_lr must be the M3-A probe's decision (0.02)"
cfg["run"]["ticket_hours"] = TICKET_HOURS
cfg["ablation"]["seconds_per_arm"] = SECONDS_PER_ARM
cfg["ablation"]["steps_per_arm"] = steps_per_arm
cfg["data"]["data_dir"] = str(DATA_DIR)
cfg_path.write_text(yaml.safe_dump(cfg))
print(cfg_path.read_text())


In [ ]:
# run_m3_ablation.py validates the plan (>=2 seeds, all arms measured,
# 8 x 1800 s <= ticket) BEFORE asking the budget, enforces its own ticket
# through scripts.train.check_ticket, and writes one ledger entry in a
# finally block whatever happens. Exit 2 = deadline truncated the matrix;
# the partial JSON is still on disk.
result = subprocess.run([
    sys.executable, "scripts/run_m3_ablation.py",
    "--config", str(cfg_path),
])
print(f"ablation exit code: {result.returncode}")
print(Path("budget/ledger.jsonl").read_text())


In [ ]:
import json
out = json.load(open(cfg["output"]["path"]))
print("complete:", out["complete"], " total_wall_seconds:", round(out.get("total_wall_seconds", float("nan"))))
for r in out["results"]:
    flag = (f"diverged@{r['diverged_at_step']}" if r["diverged_at_step"] is not None
            else "SCHEDULE INCOMPLETE" if r["schedule_incomplete"] else "ok")
    print(f"seed {r['seed']} {r['arm']:>13s}: steps={r['steps_completed']:>6d}/{r['planned_steps']} "
          f"({r['steps_per_second']:.2f}/s) trailing_flow={r['trailing_avg_loss']:.5f} [{flag}]")
print(json.dumps(out.get("paired_deltas", {}), indent=2))
